# День 4. Улучшение baseline

Цель из README: macro F1 минимум на 5% выше baseline. Все модели обучаются на одном и том же train и проверяются на одном и том же test (тот же сплит, что в дне 3).

In [12]:
import sys
sys.path.insert(0, "../src")

import json
import time
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score

from finnews_sentiment.models.train_model import build_baseline_pipeline
from finnews_sentiment.features.preprocess import preprocess_text

In [13]:
df = pd.read_csv("data/financial_clean.csv").dropna(subset=["text_clean"])

X_train, X_test, y_train, y_test = train_test_split(
    df["text_clean"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)

with open("../results/baseline_metrics.json") as f:
    baseline_f1 = json.load(f)["test_macro_f1"]
print("baseline macro F1:", baseline_f1, "| цель:", round(baseline_f1 * 1.05, 4))

baseline macro F1: 0.6809 | цель: 0.7149


In [14]:
results = []

def run(name, pipe, X_tr=X_train, X_te=X_test):
    start = time.time()
    pipe.fit(X_tr, y_train)
    pred = pipe.predict(X_te)
    f1 = f1_score(y_test, pred, average="macro")
    results.append({
        "method": name,
        "macro_f1": round(f1, 4),
        "accuracy": round(accuracy_score(y_test, pred), 4),
        "improvement_%": round((f1 - baseline_f1) / baseline_f1 * 100, 1),
        "time_sec": round(time.time() - start, 1),
    })
    print(results[-1])
    return pipe

### 0. Baseline (для контроля, должен совпасть с днём 3)

In [15]:
run("0. baseline: tfidf 5000 1-2gram + logreg", build_baseline_pipeline());

{'method': '0. baseline: tfidf 5000 1-2gram + logreg', 'macro_f1': 0.6809, 'accuracy': 0.8012, 'improvement_%': 0.0, 'time_sec': 0.3}


/Users/mak/Desktop/WORKPLACE/ML_валютная удаленка/Проекты/transformers/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


### Подход 1. Лемматизация (README)
Использую агрессивную предобработку из шаблона: лемматизация + удаление стоп-слов, цифр и пунктуации.

In [16]:
X_train_lemm = X_train.apply(preprocess_text)
X_test_lemm = X_test.apply(preprocess_text)

run("1. лемматизация + baseline", build_baseline_pipeline(), X_tr=X_train_lemm, X_te=X_test_lemm);

{'method': '1. лемматизация + baseline', 'macro_f1': 0.6785, 'accuracy': 0.7971, 'improvement_%': -0.4, 'time_sec': 0.1}


/Users/mak/Desktop/WORKPLACE/ML_валютная удаленка/Проекты/transformers/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


### Подход 2. Триграммы (README)

In [17]:
run("2. tfidf 1-3gram + logreg", Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 3))),
    ("clf", LogisticRegression(max_iter=200)),
]));

{'method': '2. tfidf 1-3gram + logreg', 'macro_f1': 0.6847, 'accuracy': 0.8065, 'improvement_%': 0.6, 'time_sec': 0.3}


### Подход 3-4. Другие модели (README): Random Forest и LinearSVC

In [18]:
run("3. tfidf + random forest", Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ("clf", RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)),
]));

run("4. tfidf + linear svc", Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ("clf", LinearSVC(max_iter=2000)),
]));

{'method': '3. tfidf + random forest', 'macro_f1': 0.6319, 'accuracy': 0.7638, 'improvement_%': -7.2, 'time_sec': 0.4}
{'method': '4. tfidf + linear svc', 'macro_f1': 0.7582, 'accuracy': 0.8117, 'improvement_%': 11.4, 'time_sec': 0.1}


### Подход 5. Веса классов
Negative всего 7%, поэтому модель почти не учится его находить. `class_weight="balanced"` делает ошибку на редком классе дороже.

In [19]:
run("5. tfidf + logreg balanced", Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
]));

run("5b. tfidf + linear svc balanced", Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ("clf", LinearSVC(max_iter=2000, class_weight="balanced")),
]));

{'method': '5. tfidf + logreg balanced', 'macro_f1': 0.7136, 'accuracy': 0.7846, 'improvement_%': 4.8, 'time_sec': 0.2}
{'method': '5b. tfidf + linear svc balanced', 'macro_f1': 0.7385, 'accuracy': 0.8012, 'improvement_%': 8.5, 'time_sec': 0.2}


### Подход 6. Слова + символьные n-граммы
Символьные n-граммы помогают с тикерами ($esi), числами, процентами и разными формами слов (rise / rose / rising).

In [20]:
def word_char_tfidf():
    return FeatureUnion([
        ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)),
    ])

run("6. word + char tfidf + logreg balanced", Pipeline([
    ("tfidf", word_char_tfidf()),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced")),
]));

{'method': '6. word + char tfidf + logreg balanced', 'macro_f1': 0.7487, 'accuracy': 0.8179, 'improvement_%': 10.0, 'time_sec': 1.4}


### Подход 7. Подбор гиперпараметра C (README)
Подбираю C кросс-валидацией только на train, чтобы не подгонять под test.

In [21]:
grid = GridSearchCV(
    Pipeline([
        ("tfidf", word_char_tfidf()),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced")),
    ]),
    {"clf__C": [0.3, 1, 3, 10]},
    scoring="f1_macro", cv=5, n_jobs=-1,
)
grid.fit(X_train, y_train)
print("лучший C:", grid.best_params_, "cv macro F1:", round(grid.best_score_, 4))

run(f"7. word + char tfidf + logreg balanced, C={grid.best_params_['clf__C']}", grid.best_estimator_);

лучший C: {'clf__C': 3} cv macro F1: 0.7666
{'method': '7. word + char tfidf + logreg balanced, C=3', 'macro_f1': 0.7631, 'accuracy': 0.8262, 'improvement_%': 12.1, 'time_sec': 1.6}


### Сравнение

In [22]:
res = pd.DataFrame(results).sort_values("macro_f1", ascending=False).reset_index(drop=True)
res

,method,macro_f1,accuracy,improvement_%,time_sec
0,"7. word + char tfidf + logreg balanced, C=3",0.7631,0.8262,12.1,1.6
1,4. tfidf + linear svc,0.7582,0.8117,11.4,0.1
2,6. word + char tfidf + logreg balanced,0.7487,0.8179,10.0,1.4
3,5b. tfidf + linear svc balanced,0.7385,0.8012,8.5,0.2
4,5. tfidf + logreg balanced,0.7136,0.7846,4.8,0.2
5,2. tfidf 1-3gram + logreg,0.6847,0.8065,0.6,0.3
6,0. baseline: tfidf 5000 1-2gram + logreg,0.6809,0.8012,0.0,0.3
7,1. лемматизация + baseline,0.6785,0.7971,-0.4,0.1
8,3. tfidf + random forest,0.6319,0.7638,-7.2,0.4


In [23]:
with open("../reports/comparison.md", "w") as f:
    f.write("| Метод | Macro F1 | Accuracy | Улучшение |\n")
    f.write("|-------|----------|----------|-----------|\n")
    for _, r in pd.DataFrame(results).iterrows():
        sign = "-" if r["method"].startswith("0.") else f"{r['improvement_%']:+.1f}%"
        f.write(f"| {r['method']} | {r['macro_f1']:.4f} | {r['accuracy']:.4f} | {sign} |\n")
print(open("../reports/comparison.md").read())

| Метод | Macro F1 | Accuracy | Улучшение |
|-------|----------|----------|-----------|
| 0. baseline: tfidf 5000 1-2gram + logreg | 0.6809 | 0.8012 | - |
| 1. лемматизация + baseline | 0.6785 | 0.7971 | -0.4% |
| 2. tfidf 1-3gram + logreg | 0.6847 | 0.8065 | +0.6% |
| 3. tfidf + random forest | 0.6319 | 0.7638 | -7.2% |
| 4. tfidf + linear svc | 0.7582 | 0.8117 | +11.4% |
| 5. tfidf + logreg balanced | 0.7136 | 0.7846 | +4.8% |
| 5b. tfidf + linear svc balanced | 0.7385 | 0.8012 | +8.5% |
| 6. word + char tfidf + logreg balanced | 0.7487 | 0.8179 | +10.0% |
| 7. word + char tfidf + logreg balanced, C=3 | 0.7631 | 0.8262 | +12.1% |



### Сохраняю лучшую модель
Если лучшей оказалась другая модель, поменяй `best` на неё.

In [24]:
best = grid.best_estimator_
best_f1 = f1_score(y_test, best.predict(X_test), average="macro")

best_results = {
    "model": f"word+char tfidf + logreg balanced, C={grid.best_params_['clf__C']}",
    "test_macro_f1": round(best_f1, 4),
    "test_accuracy": round(accuracy_score(y_test, best.predict(X_test)), 4),
    "improvement_%": round((best_f1 - baseline_f1) / baseline_f1 * 100, 1),
}
with open("../results/best_metrics.json", "w") as f:
    json.dump(best_results, f, indent=2)

joblib.dump({"model": best, "meta": best_results}, "../models/best.joblib")
joblib.dump(best.named_steps["clf"], "../models/best_model.pkl")       # как в README
joblib.dump(best.named_steps["tfidf"], "../models/vectorizer.pkl")
best_results

{'model': 'word+char tfidf + logreg balanced, C=3',
 'test_macro_f1': 0.7631,
 'test_accuracy': 0.8262,
 'improvement_%': 12.1}

## Выводы

- baseline (tfidf 5000 + logreg): macro F1 = 0.6809, цель +5% = 0.715
- лемматизация: 0.6785 (-0.4%) - не помогла, теряются отрицания (not), цифры и проценты, которые важны для тональности
- триграммы: 0.6847 (+0.6%) - почти без эффекта, тексты короткие
- random forest: 0.6319 (-7.2%) - хуже всех, деревья плохо работают с разреженным tfidf и почти не ловят редкий negative
- linear svc: 0.7582 (+11.4%) - большой прирост просто от смены модели
- веса классов (balanced): logreg 0.6809 -> 0.7136 (+4.8%), помогают ловить редкий negative (7%)
- слова + символьные n-граммы: 0.7487 (+10%) - помогают с тикерами ($esi), числами, формами слов
- подбор C через GridSearchCV на train: лучший C=3
- лучший вариант: word + char tfidf + logreg balanced, C=3 - macro F1 = 0.7631, accuracy = 0.8262, улучшение +12.1% (цель +5% выполнена)
- в test всего ~70 негативных текстов, поэтому разница между лучшими вариантами (0.75-0.76) в пределах шума
- сохранено: reports/comparison.md, results/best_metrics.json, models/best_model.pkl + vectorizer.pkl